# Markdown Reallocator: Complete Workflow Tutorial

This notebook demonstrates the complete workflow of the Markdown Reallocator library, from preprocessing to deduplication.

## Prerequisites

1. Install markdown-reallocator: `pip install markdown-reallocator`
2. Install Ollama and download embeddinggemma model
3. Ensure Ollama is running: `ollama serve`

## 1. Setup and Imports

In [ ]:
# Import necessary modules
from pathlib import Path
from markdown_reallocator.core import MarkdownPreprocessor, MarkdownSplitter, Embedder
from markdown_reallocator.modules import SearchModule, ReorderModule, DeduplicationModule
from rich import print as rprint
from rich.console import Console
from rich.table import Table

console = Console()

## 2. Load Example Document

We'll use the `llm-generated-messy.md` example which contains typical LLM formatting issues.

In [ ]:
# Load the messy example document
example_path = Path("llm-generated-messy.md")
content = example_path.read_text(encoding="utf-8")

print(f"Loaded document: {len(content)} characters")
print(f"Lines: {content.count(chr(10))}")
print(f"\nFirst 500 characters:")
print(content[:500] + "...")

## 3. Preprocessing: Fix Bold Titles

The preprocessor detects standalone bold text that should be headings.

In [ ]:
# Initialize preprocessor
preprocessor = MarkdownPreprocessor()

# Process the document
print("Processing document...")
clean_content = preprocessor.preprocess(content)

print(f"\nOriginal: {len(content)} characters")
print(f"Processed: {len(clean_content)} characters")

# Show some differences
original_lines = content.split('\n')[:30]
processed_lines = clean_content.split('\n')[:30]

console.print("\n[bold]Changes made:[/bold]")
for i, (orig, proc) in enumerate(zip(original_lines, processed_lines)):
    if orig != proc and '**' in orig:
        console.print(f"Line {i}:")
        console.print(f"  Before: {orig}")
        console.print(f"  After:  {proc}\n")

## 4. Splitting: Create Semantic Chunks

Split the document into chunks based on heading hierarchy.

In [ ]:
# Initialize splitter
splitter = MarkdownSplitter(max_tokens_per_chunk=500)

# Split the document
print("Splitting document into chunks...")
chunks = splitter.split(clean_content)

print(f"Created {len(chunks)} chunks\n")

# Display chunk information in a table
table = Table(title="Document Chunks")
table.add_column("Chunk ID", style="cyan")
table.add_column("H1 Section", style="green")
table.add_column("Length", justify="right", style="yellow")
table.add_column("Preview", style="white")

for chunk in chunks[:10]:  # Show first 10
    table.add_row(
        chunk.chunk_id,
        chunk.metadata.h1 or "(none)",
        str(len(chunk.content)),
        chunk.content[:50] + "..."
    )

console.print(table)

## 5. Embedding: Generate Vectors

Generate embeddings for each chunk using embeddinggemma.

**Note**: This requires Ollama to be running with embeddinggemma model downloaded.

In [ ]:
# Initialize embedder
print("Initializing embedder (this may take a moment)...")
embedder = Embedder(model_name="embeddinggemma")

# Generate embeddings for all chunks
print(f"Generating embeddings for {len(chunks)} chunks...")
embedded_chunks = [embedder.embed_chunk(chunk) for chunk in chunks]

print(f"\nGenerated {len(embedded_chunks)} embeddings")
print(f"Embedding dimension: {embedded_chunks[0].embedding.shape[0] if embedded_chunks[0].embedding is not None else 'None'}")

# Show first embedding
if embedded_chunks[0].embedding is not None:
    print(f"\nFirst embedding (first 10 dimensions):")
    print(embedded_chunks[0].embedding[:10])

## 6. Search: Find Similar Content

Search for chunks similar to a query.

In [ ]:
# Initialize search module
search = SearchModule(embedder=embedder, top_k=5)

# Search for RAG-related content
query = "How to build a RAG system?"
print(f"Searching for: '{query}'\n")

results = search.search(query, embedded_chunks)

# Display results
console.print(f"[bold]Found {len(results)} results:[/bold]\n")

for i, (chunk, score) in enumerate(results, 1):
    console.print(f"[bold cyan]{i}. {chunk.chunk_id}[/bold cyan] (similarity: {score:.3f})")
    console.print(f"   H1: {chunk.metadata.h1 or 'N/A'}")
    console.print(f"   Content: {chunk.content[:150]}...\n")

## 7. Reordering: Organize by Similarity

Reorder chunks to improve topical flow.

In [ ]:
# Initialize reorder module
reorder = ReorderModule()

# Reorder using sequential similarity (chain most similar)
print("Reordering chunks by sequential similarity...")
reordered = reorder.reorder_by_similarity(
    embedded_chunks,
    method="sequential",
    seed_index=0
)

print(f"Reordered {len(reordered)} chunks\n")

# Compare original vs reordered order
console.print("[bold]Original order (first 5 H1 sections):[/bold]")
for chunk in embedded_chunks[:5]:
    console.print(f"  - {chunk.metadata.h1 or '(none)'}")

console.print("\n[bold]Reordered (first 5 H1 sections):[/bold]")
for chunk in reordered[:5]:
    console.print(f"  - {chunk.metadata.h1 or '(none)'}")

## 8. Deduplication: Remove Redundancy

For this example, we'll use the `known-duplicates.md` file which contains intentional duplicates.

In [ ]:
# Load document with known duplicates
dup_path = Path("../tests/fixtures/known-duplicates.md")
dup_content = dup_path.read_text(encoding="utf-8")

# Process and embed
print("Processing document with duplicates...")
clean_dup = preprocessor.preprocess(dup_content)
dup_chunks = splitter.split(clean_dup)
dup_embedded = [embedder.embed_chunk(c) for c in dup_chunks]

print(f"Original: {len(dup_embedded)} chunks\n")

# Deduplicate
dedup = DeduplicationModule(similarity_threshold=0.95)
deduplicated = dedup.deduplicate(dup_embedded, strategy="keep_longest")

print(f"After deduplication: {len(deduplicated)} chunks")
print(f"Removed: {len(dup_embedded) - len(deduplicated)} duplicate chunks")
print(f"Reduction: {(1 - len(deduplicated)/len(dup_embedded))*100:.1f}%")

## 9. Export Results

Save the processed document back to markdown.

In [ ]:
# Reconstruct markdown from chunks
def chunks_to_markdown(chunks):
    """Convert chunks back to markdown."""
    lines = []
    for chunk in chunks:
        lines.append(chunk.content)
        lines.append("")  # Add blank line between chunks
    return "\n".join(lines)

# Save reordered version
output_path = Path("output-reordered.md")
output_content = chunks_to_markdown(reordered)
output_path.write_text(output_content, encoding="utf-8")

print(f"Saved reordered document to: {output_path}")
print(f"Size: {len(output_content)} characters")

## 10. Summary and Next Steps

In this tutorial, we covered:

1. **Preprocessing**: Fixed bold-to-heading issues
2. **Splitting**: Created semantic chunks with metadata
3. **Embedding**: Generated vector representations
4. **Search**: Found similar content with queries
5. **Reordering**: Organized chunks by similarity
6. **Deduplication**: Removed redundant content
7. **Export**: Saved processed document

### Next Steps

- Try with your own documents
- Experiment with different chunk sizes
- Test clustering-based reordering
- Use LLM-based deduplication with Gemma 2B
- Integrate into your RAG pipeline

### Performance Tips

- Enable caching for embeddings: `Embedder(cache_enabled=True)`
- Use batch processing for multiple documents
- Adjust `max_tokens_per_chunk` based on your model's context window
- Monitor GPU memory usage for large documents

### Resources

- [Documentation](https://markdown-reallocator.readthedocs.io)
- [GitHub Repository](https://github.com/seonghobae/markdown-reallocator)
- [Issue Tracker](https://github.com/seonghobae/markdown-reallocator/issues)